# W5D2 — Exploring Embedding Space — Lab

**Week 5 · Day 2 · NLP Foundations** · Lab

Yesterday's model had one column for `good` and another, unrelated column for `excellent`. A review
that used one of them told it nothing about the other. That is not a shortcoming of logistic
regression — it is what a bag of words *is*.

Today every word becomes a point in a 384-dimensional space, and you measure the distance between
points with a cosine you compute by hand in the first five minutes. `terrible` turns out to sit
beside `awful`, `paris` beside `france`, and none of that was written down by anybody.

Two things to hold on to. The famous arithmetic — `king − man + woman = queen` — **works** here, and
you will also find one that does not, which is more informative than the success. And the classifier
you fit on embeddings beats yesterday's TF-IDF **on the same 500 reviews**, then loses to it on
12,000, because more data still beats a better representation more often than anyone admits.

This is the concept the rest of the bootcamp runs on. Weeks 6 and 7 are both built on it.

<div dir="rtl" align="right">

# الأسبوع ٥ · اليوم ٢ — استكشاف فضاء التمثيل

**الأسبوع الخامس · اليوم الثاني · أساسيات معالجة اللغة** · معمل

كان لنموذج الأمس عمودٌ لـ`good` وآخر لا صلة له به لـ`excellent`. والمراجعة التي استعملت إحداهما لا
تقول له شيئًا عن الأخرى. وليس هذا نقصًا في الانحدار اللوجستي بل هو **ماهيّة** كيس الكلمات.

واليوم تصير كل كلمة نقطةً في فضاءٍ من ٣٨٤ بُعدًا، وتقيس المسافة بين النقاط بجيب تمامٍ تحسبه يدويًا
في الدقائق الخمس الأولى. فيتبيّن أن `terrible` تجلس جوار `awful`، و`paris` جوار `france`، ولم يكتب
شيئًا من ذلك أحد.

وأمران يجدر الاحتفاظ بهما. الحساب المشهور — `king − man + woman = queen` — **يعمل** هنا، وستجد
أيضًا واحدًا لا يعمل، وهو أكثر إفادةً من النجاح. والمُصنّف الذي تُدرّبه على التمثيلات يتجاوز TF-IDF
أمس **على المراجعات الخمسمئة نفسها**، ثم يخسر أمامه على الاثنتي عشرة ألفًا، لأن كثرة البيانات ما
زالت تتجاوز حُسن التمثيل أكثر ممّا يعترف أحد.

وهذا هو المفهوم الذي تعمل عليه بقيّة المعسكر. فالأسبوعان السادس والسابع مبنيّان عليه.

</div>

> **This is your lab notebook.** Work through the hints — they tell you what to do and where
> to look, not what to type. Stuck for more than ten minutes on one task? Open the `_guided`
> version. That is not cheating; sitting stuck in silence is the only mistake. The full
> solution is released at the end of the day.

<div dir="rtl" align="right">

> **هذا دفتر المعمل الخاص بك.** اعمل وفق الإرشادات — فهي تخبرك بما يجب فعله وأين تبحث، لا بما
> تكتبه حرفيًا. إذا توقّفت أكثر من عشر دقائق عند مهمة واحدة فافتح نسخة `_guided`؛ هذا ليس غشًّا،
> والخطأ الوحيد هو أن تبقى متوقّفًا بصمت. ويُنشر الحل الكامل في نهاية اليوم.

</div>

## Learning objectives

By the end of this lab you can:

- Implement cosine similarity and predict what scaling a vector does to it, before running it.
- Check whether a set of vectors is normalised, and say what follows for the dot product if it is.
- Return the nearest neighbours of a term and read the list critically rather than approvingly.
- Run semantic arithmetic, and report a case where it fails.
- Say what a 2-D projection of a 384-dimensional space is, and is not, evidence of.
- Measure a bias in an embedding space and state its consequence for a downstream model.
- Compare an embedding classifier against yesterday's baseline on identical data and folds.

<div dir="rtl" align="right">

## أهداف التعلّم

في نهاية هذا المعمل تستطيع:

- أن تُنفّذ تشابه جيب التمام وتتنبّأ بما يفعله قياس متّجهٍ به قبل التشغيل.
- أن تفحص هل مجموعة متّجهات مُطبَّعة، وأن تقول ما يلزم للجداء القياسي إن كانت.
- أن تُعيد الجيران الأقرب لمصطلح وأن تقرأ القائمة نقديًا لا استحسانًا.
- أن تُجري حسابًا دلاليًا، وأن تعرض حالةً يفشل فيها.
- أن تقول على ماذا يدلّ إسقاطٌ ثنائي الأبعاد لفضاءٍ من ٣٨٤ بُعدًا وعلى ماذا لا يدلّ.
- أن تقيس تحيّزًا في فضاء تمثيل وأن تُبيّن أثره في نموذجٍ لاحق.
- أن تقارن مُصنّف تمثيلات بأساس الأمس على بياناتٍ وأثلامٍ متطابقة.

</div>


## About the data

**Two datasets, and only one of them is big.**

`sentence_embeddings_sample` — 50,000 English tokens, each with a 384-dimensional vector produced by
`sentence-transformers/all-MiniLM-L6-v2`. Columns: `token`, `rank` (1 is the most frequent word in
the source list) and `embedding`. The vectors are **L2-normalised**, which is task 2.1's first
observation and simplifies everything after it. It is precomputed so that this lab spends its ninety
minutes on the space rather than on a progress bar; the token list is frequency-ranked so that the
neighbours you read are words you recognise.

`reviews_sentiment` — yesterday's 12,000 reviews, unchanged, so today's comparison against
`tfidf_baseline.json` is a comparison and not an anecdote.

**The known problem: these vectors carry the biases of the corpus they were trained on.** Task 2.5
measures one of them — the same measurement that has been reproduced on every embedding model since
2016, on a model released in 2021. It is not a flaw you can patch out with a better random seed, and
the task asks you to write down what it means for a model you put downstream of it.

**First-run download:** tasks 2.6 and the stretch load MiniLM itself to embed sentences, about 90 MB
the first time, then cached. Everything before task 2.6 works offline from the parquet file.

<div dir="rtl" align="right">

## عن البيانات

**مجموعتان، وواحدة منهما فقط كبيرة.**

`sentence_embeddings_sample` — خمسون ألف رمز إنجليزي، لكلٍّ منها متّجه بـ٣٨٤ بُعدًا أنتجه
`sentence-transformers/all-MiniLM-L6-v2`. والأعمدة: `token` و`rank` (والواحد أكثر كلمة تكرارًا في
قائمة المصدر) و`embedding`. والمتّجهات **مُطبَّعة بمعيار L2**، وهذه أول ملاحظة في المهمة ٢٫١ وهي
تُبسّط كل ما بعدها. وهي مُحسَّبة مسبقًا ليُنفق هذا المعمل تسعين دقيقته على الفضاء لا على شريط تقدّم،
وقائمة الرموز مرتّبة بالتكرار ليكون الجيران الذين تقرأهم كلماتٍ تعرفها.

`reviews_sentiment` — مراجعات الأمس الاثنتا عشرة ألفًا كما هي، فتكون مقارنة اليوم بـ
`tfidf_baseline.json` مقارنةً لا حكايةً.

**والمشكلة المعروفة: تحمل هذه المتّجهات تحيّزات المُدوّنة التي دُرِّبت عليها.** وتقيس المهمة ٢٫٥
واحدًا منها — وهو القياس نفسه الذي أُعيد إنتاجه على كل نموذج تمثيل منذ سنة ٢٠١٦، على نموذج صدر سنة
٢٠٢١. وليس خللًا تُرقّعه ببذرة عشوائية أفضل، وتطلب منك المهمة أن تكتب ما يعنيه لنموذجٍ تضعه بعده.

**تنزيل أول مرّة:** تُحمّل المهمة ٢٫٦ وقسم التمديد نموذج MiniLM نفسه لتمثيل الجمل، نحو ٩٠ ميغابايت
في المرّة الأولى ثم يُخزَّن. وكل ما قبل المهمة ٢٫٦ يعمل بلا شبكة من ملف الباركيه.

</div>


## Setup

The embedding file is 78 MB and loads in a couple of seconds. `np.stack` on the `embedding` column
turns it into a `(50000, 384)` array once, and everything after that is one matrix multiplication.

<div dir="rtl" align="right">

## الإعداد

ملف التمثيلات ٧٨ ميغابايت ويُحمّل في ثانيتين. و`np.stack` على عمود `embedding` يحوّله مرّةً واحدة
إلى مصفوفة `(50000, 384)`، وكل ما بعد ذلك جداء مصفوفات واحد.

</div>


In [ ]:
# === AIEP portable setup — works locally (conda) and on Google Colab ===============
try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, device, versions
from aiep.data import get_dataset, describe_dataset, load_artefact
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, check_close, check_shape, report
from aiep.viz import use_course_style, savefig

ensure("scikit-learn", "matplotlib", "sentence-transformers")
seed_everything(42)

import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

use_course_style()
np.set_printoptions(precision=4, suppress=True)

# NumPy built against Apple's Accelerate framework raises spurious floating-point flags inside
# `matmul`. `aiep` already filters them for scikit-learn's frames; this lab does its own matrix
# products over 50,000 rows, so the same filter is applied here. See shared/aiep/__init__.py.
import warnings

warnings.filterwarnings("ignore", message=".*encountered in matmul", category=RuntimeWarning)

SEED = 42
N_SPLITS = 5
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# The two vectors from this morning's slide.
A = np.array([3.0, 4.0])
B = np.array([4.0, 3.0])
PERPENDICULAR = np.array([-4.0, 3.0])

embeddings = pd.read_parquet(get_dataset("sentence_embeddings_sample"))
reviews = pd.read_parquet(get_dataset("reviews_sentiment"))

E = np.stack(embeddings.embedding.to_numpy())
INDEX = {token: i for i, token in enumerate(embeddings.token)}

print(describe_dataset("sentence_embeddings_sample"))
print(f"\nembeddings {E.shape} | reviews {reviews.shape}")
print(f"most frequent tokens: {embeddings.token.head(8).tolist()}")
print(versions(), "| device:", device())

## Section 1 — Warm-up: three cosines  (≈25 min)

Everything here works. The formula is the dot product over the product of the norms, and you write
it yourself — three lines, no library.

Three cases, all from this morning:

1. `a = [3, 4]`, `b = [4, 3]` → `24 / (5 × 5)` = **0.96**. Nearly the same direction.
2. `a` against `b × 10` → `240 / (5 × 50)` = **0.96**. **Exactly** the same number. The magnitude
   cancels, which is the entire reason cosine is used on text: a long document and a short one about
   the same subject point the same way.
3. `a = [3, 4]` against `[-4, 3]` → **0.0**. Perpendicular, and the two share no direction at all.

Change the numbers and watch which of the three answers moves.

<div dir="rtl" align="right">

## القسم الأول — الإحماء: ثلاثة جيوب تمام (نحو ٢٥ دقيقة)

كل ما هنا يعمل. والصيغة هي الجداء القياسي على جداء المعياريْن، وتكتبها بنفسك — ثلاثة أسطر بلا
مكتبة.

ثلاث حالات، كلها من هذا الصباح:

١. `a = [3, 4]` و`b = [4, 3]` ← `24 / (5 × 5)` = **٠٫٩٦**. اتّجاهٌ يكاد يكون واحدًا.
٢. `a` مقابل `b × 10` ← `240 / (5 × 50)` = **٠٫٩٦**. العدد نفسه **تمامًا**. فيتلاشى المقدار، وهذا
   هو سبب استخدام جيب التمام على النصوص كله: فالمستند الطويل والقصير في الموضوع نفسه يشيران إلى
   الجهة نفسها.
٣. `a = [3, 4]` مقابل `[-4, 3]` ← **٠٫٠**. متعامدان، ولا يتشاركان اتّجاهًا أصلًا.

غيّر الأعداد وراقب أيّ الأجوبة الثلاثة يتحرّك.

</div>


In [ ]:
def cosine(u, v):
    """Cosine similarity: the dot product over the product of the norms."""
    return float(u @ v / (np.linalg.norm(u) * np.linalg.norm(v)))


COSINE_AB = cosine(A, B)
COSINE_SCALED = cosine(A, B * 10)
COSINE_PERPENDICULAR = cosine(A, PERPENDICULAR)

print(f"cosine({A}, {B})      = {COSINE_AB:.4f}   (slide: 0.96)")
print(f"cosine({A}, {B * 10}) = {COSINE_SCALED:.4f}   (slide: 0.96 — unchanged)")
print(f"cosine({A}, {PERPENDICULAR})    = {COSINE_PERPENDICULAR:.4f}   (slide: 0.0)")
print(f"\nscaling changed the answer by {abs(COSINE_AB - COSINE_SCALED):.1e}")
print(f"cosine(a, a) = {cosine(A, A):.4f} and cosine(a, -a) = {cosine(A, -A):.4f} — the range "
      f"is [-1, 1] and both ends are reachable")

## Section 2 — Core: six tasks  (≈60 min)

1. Load the space, check the shape, and check whether the vectors are normalised.
2. Nearest neighbours for five terms — and read them critically.
3. Semantic arithmetic: one that works, and find one that does not.
4. PCA to two dimensions, 200 labelled points, and what the picture does not prove.
5. The bias measurement, and its consequence for a downstream model.
6. Embeddings against yesterday's baseline, on the same reviews and the same folds.

<div dir="rtl" align="right">

## القسم الثاني — الأساسي: ست مهام (نحو ٦٠ دقيقة)

١. حمّل الفضاء، وافحص الشكل، وافحص هل المتّجهات مُطبَّعة.
٢. الجيران الأقرب لخمسة مصطلحات — واقرأهم نقديًا.
٣. الحساب الدلالي: واحد يعمل، وجِد واحدًا لا يعمل.
٤. تحليل المكوّنات الرئيسة إلى بُعدين، ومئتا نقطة مُسمّاة، وما لا تُبرهنه الصورة.
٥. قياس التحيّز، وأثره في نموذجٍ لاحق.
٦. التمثيلات مقابل أساس الأمس، على المراجعات نفسها والأثلام نفسها.

</div>


### Task 2.1 — the shape, and one question about it

Print the shape of the matrix — `(50000, 384)`, fifty thousand tokens in a 384-dimensional space.

Then answer the question that decides how you write every line after this one: **are these vectors
normalised?** Compute the L2 norm of every row and look at the minimum and the maximum.

They are, to within floating-point noise. Say what follows: if `‖u‖ = ‖v‖ = 1`, the cosine formula's
denominator is `1`, so `u @ v` **is** the cosine. Every similarity in the rest of this lab is one
matrix multiplication, and the 50,000-row search in task 2.2 takes milliseconds instead of a loop.

<div dir="rtl" align="right">

### المهمة ٢٫١ — الشكل وسؤال واحد عنه

اطبع شكل المصفوفة — `(50000, 384)`، خمسون ألف رمز في فضاءٍ من ٣٨٤ بُعدًا.

ثم أجب السؤال الذي يقرّر كيف تكتب كل سطر بعد هذا: **هل هذه المتّجهات مُطبَّعة؟** احسب معيار L2 لكل
صف وانظر الأصغر والأكبر.

هي مُطبَّعة إلى حدّ ضجيج الفاصلة العائمة. وقل ما يلزم من ذلك: إن كان `‖u‖ = ‖v‖ = 1` فمقام صيغة جيب
التمام واحد، فيكون `u @ v` **هو** جيب التمام. فتصير كل تشابهات بقيّة المعمل جداء مصفوفات واحدًا،
ويأخذ البحث في الخمسين ألف صف في المهمة ٢٫٢ أجزاءً من الثانية بدل حلقة.

</div>


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) np.linalg.norm with axis=1 gives you one norm per row.
# 2) Print the min and the max, not the mean — a mean of 1.0 hides a mixture.
# 3) Then verify the consequence on one pair: compare u @ v against your cosine(u, v).
# Search: "numpy linalg norm axis rows unit vectors"
# https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html
#
# ١) `np.linalg.norm` بـ`axis=1` يعطيك معيارًا لكل صف.
# ٢) اطبع الأصغر والأكبر لا المتوسط — فمتوسط ١٫٠ يُخفي خليطًا.
# ٣) ثم تحقّق من اللازم على زوج واحد: قارن `u @ v` بـ`cosine(u, v)` عندك.
# ابحث عن: "numpy linalg norm axis rows unit vectors"
# https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html
# ────────────────────────────────────────────────────────────────────

# TODO: that the dot product equals your cosine when the vectors are unit length.
# مهمة: جيب التمام عندك حين تكون المتّجهات وحدوية الطول.

### Task 2.2 — nearest neighbours, read critically

For each of five query terms — `good`, `terrible`, `paris`, `doctor`, `phone` — return the ten
nearest tokens by cosine, excluding the query itself.

Then read them properly, which means looking for what is *wrong* with each list:

- `good` returns `excellent`, `great`, `nice` — and `goody`. That last one is not a synonym; it is a
  string that shares five characters. A subword model will do that.
- `terrible` returns `horrible`, `awful`, `dreadful` — and `terribly`. Same story: an inflection, not
  a sibling.
- `paris` returns `parisian`, `france`, `seine`, `louvre`. Note that these are not synonyms at all —
  they are *associations*. Embedding space measures "appears in similar contexts", which is neither
  synonymy nor relatedness but overlaps with both.

Write one sentence on the difference between "similar" and "appears in similar contexts". Every
retrieval system in week 7 depends on you having that distinction straight.

<div dir="rtl" align="right">

### المهمة ٢٫٢ — الجيران الأقرب، مقروئين نقديًا

لكلٍّ من خمسة مصطلحات — `good` و`terrible` و`paris` و`doctor` و`phone` — أعِد أقرب عشرة رموز بجيب
التمام، مستثنيًا المُستعلَم نفسه.

ثم اقرأها قراءةً صحيحة، ومعناها البحث عمّا هو **خاطئ** في كل قائمة:

- يُعيد `good` كلمات `excellent` و`great` و`nice` — و`goody`. وليست الأخيرة مرادفًا بل سلسلةً
  تتشارك خمسة محارف. والنموذج القائم على الرموز الجزئية يفعل ذلك.
- ويُعيد `terrible` كلمات `horrible` و`awful` و`dreadful` — و`terribly`. والحكاية نفسها: تصريفٌ لا
  شقيق.
- ويُعيد `paris` كلمات `parisian` و`france` و`seine` و`louvre`. ولاحظ أنها ليست مرادفات أصلًا بل
  **اقترانات**. فيقيس فضاء التمثيل «الورود في سياقات متشابهة»، وهذا ليس ترادفًا ولا صلةً لكنه
  يتقاطع معهما.

اكتب جملةً واحدة عن الفرق بين «متشابه» و«يرد في سياقات متشابهة». فكل نظام استرجاع في الأسبوع
السابع يتوقّف على استقامة هذا التمييز عندك.

</div>


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Because the rows are unit length, the similarity of one token against all 50,000 is
#    E @ E[i] — one matrix-vector product, no loop.
# 2) np.argsort on the negated similarities gives you the best first. Or look up argpartition.
# 3) Drop the query itself from the result — it will always be first with a similarity
#    of 1.0, and a neighbour list that includes the query is not a neighbour list.
# 4) Return (token, similarity) pairs so you can print the numbers, not just the words.
# Search: "numpy top k cosine similarity argsort argpartition"
# https://numpy.org/doc/stable/reference/generated/numpy.argpartition.html
#
# ١) لأن الصفوف وحدوية الطول فتشابه رمزٍ مع الخمسين ألفًا كلها هو `E @ E[i]` — جداء
#    مصفوفة في متّجه، بلا حلقة.
# ٢) و`np.argsort` على التشابهات المنفيّة يعطيك الأفضل أولًا. أو راجع `argpartition`.
# ٣) أسقط المُستعلَم نفسه من الناتج — فسيكون أولًا دائمًا بتشابه ١٫٠، وقائمة جيرانٍ فيها
#    المُستعلَم ليست قائمة جيران.
# ٤) أعِد أزواج `(token, similarity)` لتطبع الأعداد لا الكلمات وحدها.
# ابحث عن: "numpy top k cosine similarity argsort argpartition"
# https://numpy.org/doc/stable/reference/generated/numpy.argpartition.html
# ────────────────────────────────────────────────────────────────────

QUERY_TERMS = ["good", "terrible", "paris", "doctor", "phone"]
    # TODO: (token, similarity) pairs.
    # مهمة: أزواج `(token, similarity)`.
# TODO: the lists in NEIGHBOURS for the report.
# مهمة: للتقرير.

### Task 2.3 — arithmetic, and the one that fails

The famous one first: take the vector for `king`, subtract `man`, add `woman`, and find the nearest
tokens to the result — excluding the three inputs, which will otherwise dominate. `queen` comes
back first. It is a real result and it deserves the reaction it gets.

Now do your own, and **report the ones that fail**. Two are provided to get you started:

- `paris − france + italy` should give `rome`. It gives **`florence`**, with `italia`, `milano` and
  `venice` behind it. The analogy did not fail *randomly* — it landed on the right kind of thing, an
  Italian city, and missed the specific relation "is the capital of".
- `better − good + bad` gives `worse`, and that one works.

The failures are the informative half. A vector arithmetic that lands on the right category and the
wrong member tells you the space encodes topical association much more strongly than it encodes
specific relations — and that is precisely the failure mode you will spend week 7 designing around.

<div dir="rtl" align="right">

### المهمة ٢٫٣ — الحساب، والذي يفشل منه

المشهور أولًا: خُذ متّجه `king`، واطرح `man`، وأضف `woman`، وجِد أقرب الرموز إلى الناتج — مستثنيًا
المداخل الثلاثة، وإلّا هيمنت. فتعود `queen` أولًا. وهي نتيجة حقيقية تستحقّ ما تلقاه من ردّ فعل.

والآن اصنع حسابك، و**اعرض ما يفشل منه**. واثنان مُعطيان لتبدأ:

- `paris − france + italy` يجب أن يُعطي `rome`. ويُعطي **`florence`**، وخلفها `italia` و`milano`
  و`venice`. ولم يفشل التناظر **عشوائيًا** بل وقع على النوع الصحيح، مدينةٍ إيطالية، وأخطأ العلاقة
  المحدّدة «عاصمةُ».
- و`better − good + bad` يُعطي `worse`، وهذا يعمل.

والإخفاقات هي النصف المُفيد. فحساب متّجهات يقع على الفئة الصحيحة والعضو الخاطئ يقول لك إن الفضاء
يُرمّز الاقتران الموضوعي أقوى بكثير ممّا يُرمّز العلاقات المحدّدة — وهذا بعينه نمط الإخفاق الذي
تُنفق الأسبوع السابع في التصميم حوله.

</div>


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) The result vector is E[a] - E[b] + E[c]. Normalise it before scoring, or the
#    magnitudes of the three inputs leak into the ranking.
# 2) Exclude a, b and c from the candidates. Skipping this step is why this trick looks
#    better in blog posts than it is.
# 3) Print the top 5 for each analogy with the expected answer next to it, and mark
#    whether the expected answer came first, came in the top 5, or did not appear.
# Search: "word analogy vector arithmetic exclude query words"
# https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html
#
# ١) متّجه الناتج هو `E[a] - E[b] + E[c]`. طبّعه قبل التقييم، وإلّا تسرّبت مقادير
#    المداخل الثلاثة إلى الترتيب.
# ٢) استثنِ `a` و`b` و`c` من المرشّحين. وتخطّي هذه الخطوة هو سبب كون الحيلة في المدوّنات
#    أجمل منها في الحقيقة.
# ٣) اطبع أفضل خمسة لكل تناظر وبجانبها الجواب المتوقّع، وأشِر هل جاء الجواب المتوقّع
#    أولًا، أم في الخمسة الأولى، أم لم يظهر.
# ابحث عن: "word analogy vector arithmetic exclude query words"
# https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html
# ────────────────────────────────────────────────────────────────────

ANALOGIES = [
    ("king", "man", "woman", "queen"),
    ("paris", "france", "italy", "rome"),
    ("better", "good", "bad", "worse"),
]
    # TODO: top k tokens that are not one of the three inputs.
    # مهمة: الثلاثة.
# TODO: failed in ANALOGY_FAILURES.
# مهمة: `ANALOGY_FAILURES`.

### Task 2.4 — 200 points in two dimensions, and what that is not

Take 200 tokens — the provided list groups them by theme so the picture is readable — run PCA down
to two components, and scatter them with their labels.

Related words land near each other. It looks like proof of something.

It is not. Two components out of 384 retain a **small fraction** of the variance; print the
`explained_variance_ratio_` sum and read it out loud. Two words far apart in the picture may be
neighbours in the space, and two words touching in the picture may be nowhere near each other.

The plot is a **communication device**, not evidence. When you need to know whether two tokens are
close, compute the cosine in 384 dimensions — which is one line, and which is what task 2.2 did.
Write that sentence down; someone will show you a t-SNE plot as an argument within the year.

<div dir="rtl" align="right">

### المهمة ٢٫٤ — مئتا نقطة في بُعدين، وما ليست دليلًا عليه

خُذ مئتَي رمز — والقائمة المُعطاة تجمعها بالموضوع لتكون الصورة مقروءة — وشغّل تحليل المكوّنات
الرئيسة إلى مكوّنين، وانثرها مع تسمياتها.

فتقع الكلمات المتصلة قرب بعضها. ويبدو ذلك برهانًا على شيء.

وليس كذلك. فمكوّنان من ٣٨٤ يحفظان **كسرًا صغيرًا** من التباين؛ اطبع مجموع
`explained_variance_ratio_` واقرأه بصوتٍ عالٍ. فقد تكون كلمتان متباعدتان في الصورة جارتين في
الفضاء، وقد تكون كلمتان متلامستان في الصورة غير متقاربتين أصلًا.

فالرسم **أداة إبلاغ** لا دليل. وحين تريد أن تعرف هل رمزان قريبان فاحسب جيب التمام في ٣٨٤ بُعدًا —
وهو سطر واحد، وهو ما فعلته المهمة ٢٫٢. اكتب تلك الجملة؛ فسيُريك أحدهم رسم t-SNE حجّةً خلال سنة.

</div>


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Build the index list from THEMES, then slice E with it — one array of 200 rows.
# 2) PCA(n_components=2, random_state=42).fit_transform, then scatter, colouring by theme.
# 3) Print pca.explained_variance_ratio_.sum(). That number is the point of the task.
# 4) Annotate a subset of the points — 200 labels on one axis is unreadable.
# Search: "sklearn PCA explained_variance_ratio_ scatter annotate"
# https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html
#
# ١) ابنِ قائمة الفهارس من `THEMES`، ثم اقتطع `E` بها — مصفوفة واحدة بمئتَي صف.
# ٢) `PCA(n_components=2, random_state=42).fit_transform` ثم انثر ولوّن بالموضوع.
# ٣) اطبع `pca.explained_variance_ratio_.sum()`. وذلك العدد هو مقصد المهمة.
# ٤) وسمِّ مجموعةً جزئية من النقاط — فمئتا تسمية على محورٍ واحد غير مقروءة.
# ابحث عن: "sklearn PCA explained_variance_ratio_ scatter annotate"
# https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html
# ────────────────────────────────────────────────────────────────────

from sklearn.decomposition import PCA
THEMES = {
    "sentiment": ["good", "great", "excellent", "wonderful", "terrible", "awful",
                 "horrible", "bad", "poor", "mediocre", "brilliant", "dreadful",
                 "lovely", "nasty", "superb", "amazing", "fantastic", "dismal", "lousy",
                 "pleasant", "disappointing", "outstanding", "rubbish", "charming",
                 "appalling", "delightful", "atrocious", "splendid", "grim", "marvellous",
                 "shoddy", "exquisite", "abysmal", "decent", "horrid", "gorgeous",
                 "crummy", "sublime", "bland", "glorious"],
    "places": ["paris", "france", "rome", "italy", "berlin", "germany", "madrid",
              "spain", "london", "england", "tokyo", "japan", "cairo", "egypt",
              "moscow", "russia", "lisbon", "portugal", "vienna", "austria", "athens",
              "greece", "dublin", "ireland", "oslo", "norway", "warsaw", "poland",
              "ankara", "turkey", "beirut", "lebanon", "amman", "jordan", "tunis",
              "tunisia", "casablanca", "morocco", "baghdad", "iraq"],
    "people": ["doctor", "nurse", "teacher", "engineer", "lawyer", "farmer", "pilot",
              "soldier", "waiter", "driver", "scientist", "chef", "dentist", "plumber",
              "banker", "artist", "singer", "writer", "painter", "builder", "tailor",
              "barber", "butcher", "baker", "cleaner", "guard", "clerk", "cashier",
              "miner", "sailor", "mechanic", "architect", "librarian", "journalist",
              "photographer", "accountant", "electrician", "carpenter", "surgeon",
              "pharmacist"],
    "devices": ["phone", "camera", "laptop", "printer", "keyboard", "monitor", "speaker",
               "battery", "charger", "headphones", "tablet", "router", "microphone",
               "mouse", "scanner", "projector", "console", "modem", "webcam", "amplifier",
               "antenna", "stereo", "cable", "joystick", "radio", "television",
               "recorder", "player", "watch", "clock", "calculator", "fan", "heater",
               "kettle", "toaster", "blender", "vacuum", "lamp", "radiator", "freezer"],
    "food": ["pizza", "bread", "cheese", "coffee", "chocolate", "rice", "chicken",
            "salad", "soup", "apple", "banana", "butter", "pasta", "sugar",
            "honey", "lemon", "orange", "pepper", "garlic", "onion", "potato",
            "tomato", "carrot", "spinach", "yoghurt", "cream", "bacon", "sausage",
            "steak", "salmon", "shrimp", "noodles", "cereal", "biscuit", "cookie",
            "pancake", "waffle", "pudding", "mustard", "vinegar"],
}
# TODO: some labels, and print how much variance the two components actually keep.
# مهمة: واطبع كم من التباين يحفظه المكوّنان فعلًا.

### Task 2.5 — measure the bias, then say what it does

Four term sets are provided: two groups of pronouns and kinship terms, and two attribute sets —
one about careers and one about home and family.

Compute the mean vector of each set, then the cosine between each group and each attribute set, and
print the four numbers with the two gaps.

Read what comes out honestly, because it is not the tidy result the textbook version promises:

- On the **career** attributes the two groups come out within about a hundredth of each other. There
  is essentially no gap. If you had only run this half you would have concluded the space is clean.
- On the **family** attributes there is a gap of several hundredths, in the direction the literature
  reports, and it is stable across seeds because nothing here is random.

So one of the two measurements reproduces the published effect and the other does not. Report both.
Then write **two sentences on the consequence for a downstream model** — technical, not moral. The
question to answer is: if you fit a classifier on top of these vectors and one class correlates with
the family attributes, what does the model learn, and what happens to the group whose mean vector
sits closer to them?

<div dir="rtl" align="right">

### المهمة ٢٫٥ — قِس التحيّز ثم قل ما يفعله

أربع مجموعات مصطلحات مُعطاة: مجموعتان من الضمائر وأسماء القرابة، ومجموعتا سماتٍ — إحداهما عن
المِهَن والأخرى عن البيت والأسرة.

احسب متّجه المتوسط لكل مجموعة، ثم جيب التمام بين كل مجموعةٍ وكل مجموعة سمات، واطبع الأعداد الأربعة
مع الفجوتين.

واقرأ ما يخرج بأمانة، فليس هو النتيجة المرتّبة التي يَعِد بها الكتاب:

- على سمات **المِهَن** تخرج المجموعتان في حدود جزءٍ من مئة إحداهما من الأخرى. فلا فجوة تقريبًا. ولو
  شغّلت هذا النصف وحده لاستنتجت أن الفضاء نظيف.
- وعلى سمات **الأسرة** توجد فجوة بعدّة أجزاء من مئة، في الاتجاه الذي تعرضه الأدبيات، وهي مستقرّة
  عبر البذور لأن لا شيء هنا عشوائي.

فأحد القياسين يُعيد إنتاج الأثر المنشور والآخر لا. اعرض الاثنين. ثم اكتب **جملتين عن الأثر في
نموذجٍ لاحق** — تقنيتين لا أخلاقيتين. والسؤال المطلوب جوابه: إن درّبت مُصنّفًا على هذه المتّجهات
وارتبطت إحدى الفئات بسمات الأسرة، فماذا يتعلّم النموذج، وما يحدث للمجموعة التي يجلس متّجه متوسّطها
أقرب إليها؟

</div>


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) The mean vector of a set is E[[INDEX[w] for w in words]].mean(axis=0). It is not
#    unit length any more, so use your cosine function rather than a dot product.
# 2) Four cosines, two gaps. Print all six numbers.
# 3) Nothing here is random — run it twice and confirm you get the same numbers, so the
#    gap you report is a property of the space and not of a seed.
# 4) Then write the two sentences. The question is mechanical: what does a classifier
#    fit on these vectors do with a group that sits closer to one attribute set?
# Search: "WEAT word embedding association test mean cosine"
# https://scikit-learn.org/stable/modules/metrics.html#cosine-similarity
#
# ١) متّجه متوسط المجموعة هو `E[[INDEX[w] for w in words]].mean(axis=0)`. ولم يبقَ
#    وحدوي الطول، فاستخدم دالة جيب التمام عندك لا الجداء القياسي.
# ٢) أربعة جيوب تمام وفجوتان. اطبع الأعداد الستة كلها.
# ٣) ولا شيء هنا عشوائي — شغّله مرّتين وتأكّد أنك تحصل على الأعداد نفسها، لتكون الفجوة
#    التي تعرضها خاصّية الفضاء لا خاصّية بذرة.
# ٤) ثم اكتب الجملتين. والسؤال آليّ: ماذا يفعل مُصنّف مُدرَّب على هذه المتّجهات بمجموعةٍ
#    تجلس أقرب إلى إحدى مجموعتَي السمات؟
# ابحث عن: "WEAT word embedding association test mean cosine"
# https://scikit-learn.org/stable/modules/metrics.html#cosine-similarity
# ────────────────────────────────────────────────────────────────────

GROUP_A = ["he", "him", "his", "man", "men", "boy", "father", "brother", "son", "husband"]
GROUP_B = ["she", "her", "hers", "woman", "women", "girl", "mother", "sister",
           "daughter", "wife"]
CAREER = ["career", "salary", "office", "business", "engineer", "manager",
          "professional", "executive"]
FAMILY = ["family", "home", "children", "marriage", "household", "parents",
          "wedding", "kitchen"]
# TODO: result in BIAS_MEASUREMENT.
# مهمة: `BIAS_MEASUREMENT`.

### Task 2.6 — against yesterday's baseline, on the same data

Now use the space for something. Embed 500 reviews with MiniLM, fit `LogisticRegression` on the
embeddings, and cross-validate on `StratifiedKFold(5, shuffle=True, random_state=42)`.

Then make the comparison **fair**, which takes one extra line: run yesterday's TF-IDF pipeline on
**the same 500 reviews and the same folds**. Yesterday's headline number came from 12,000 reviews,
and comparing a 500-review embedding model against a 12,000-review TF-IDF model measures the size of
the training set, not the quality of the representation.

Report three numbers: embeddings on 500, TF-IDF on 500, and yesterday's TF-IDF on 12,000. You will
find the embeddings win by a wide margin on equal data — and lose to yesterday's number, because
24× the data is worth more than 384 dimensions of pretraining. Both facts belong in the report.

Then the payoff: classify yesterday's negation pair. `"the service was good"` and
`"the service was not good"` are 0.8 apart in cosine — not identical, the way TF-IDF made them — and
the embedding model separates them for a reason that is not "the token `not` is suspicious".

<div dir="rtl" align="right">

### المهمة ٢٫٦ — مقابل أساس الأمس، على البيانات نفسها

واستخدم الفضاء الآن في شيء. مثّل خمسمئة مراجعة بـMiniLM، ودرّب `LogisticRegression` على
التمثيلات، وتحقّق تقاطعيًا بـ`StratifiedKFold(5, shuffle=True, random_state=42)`.

ثم اجعل المقارنة **عادلة**، وهذا يأخذ سطرًا إضافيًا: شغّل خطّ TF-IDF أمس على **المراجعات الخمسمئة
نفسها والأثلام نفسها**. فرقم الأمس الرئيس جاء من اثنتي عشرة ألف مراجعة، ومقارنة نموذج تمثيلات على
خمسمئة بنموذج TF-IDF على اثنتي عشرة ألفًا تقيس حجم بيانات التدريب لا جودة التمثيل.

اعرض ثلاثة أرقام: التمثيلات على خمسمئة، وTF-IDF على خمسمئة، وTF-IDF أمس على اثنتي عشرة ألفًا.
وستجد التمثيلات تفوز بفارق واسع على بياناتٍ متساوية — وتخسر أمام رقم الأمس، لأن أربعةً وعشرين ضعفًا
من البيانات أثمن من ٣٨٤ بُعدًا من التدريب المسبق. والحقيقتان كلتاهما من التقرير.

ثم المكسب: صنّف زوج النفي أمس. فـ`"the service was good"` و`"the service was not good"` بينهما نحو
٠٫٨ في جيب التمام — لا متطابقتان كما جعلهما TF-IDF — ويفصلهما نموذج التمثيل لسببٍ غير أن الرمز
`not` مشبوه.

</div>


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) SentenceTransformer(MODEL_NAME).encode(list_of_strings, normalize_embeddings=True)
#    gives you a (n, 384) array. 500 reviews takes a few seconds on a CPU.
# 2) Sample the 500 with a fixed random_state so the folds are reproducible, then build
#    ONE StratifiedKFold and pass it to both cross_val_score calls.
# 3) load_artefact("tfidf_baseline.json") finds yesterday's file, falling back to
#    shared/solutions_cache if you did not run D1.
# 4) For the negation pair, fit the embedding classifier on all 500 first, then predict.
# Search: "sentence-transformers encode logistic regression cross_val_score"
# https://www.sbert.net/docs/quickstart.html
#
# ١) `SentenceTransformer(MODEL_NAME).encode(list_of_strings, normalize_embeddings=True)`
#    يعطيك مصفوفة `(n, 384)`. وخمسمئة مراجعة تأخذ ثوانٍ على المعالج.
# ٢) انتقِ الخمسمئة بـ`random_state` ثابت لتكون الأثلام قابلة لإعادة الإنتاج، ثم ابنِ
#    `StratifiedKFold` **واحدًا** ومرّره إلى نداءَي `cross_val_score`.
# ٣) و`load_artefact("tfidf_baseline.json")` يجد ملف الأمس، ويرجع إلى
#    `shared/solutions_cache` إن لم تُشغّل اليوم الأول.
# ٤) ولزوج النفي، درّب مُصنّف التمثيلات على الخمسمئة كلها أولًا ثم تنبّأ.
# ابحث عن: "sentence-transformers encode logistic regression cross_val_score"
# https://www.sbert.net/docs/quickstart.html
# ────────────────────────────────────────────────────────────────────

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
NEGATION_PAIR = ["the service was good", "the service was not good"]
SUBSET_SIZE = 500
yesterday = json.loads(load_artefact("tfidf_baseline.json").read_text(encoding="utf-8"))
print(f"yesterday on {yesterday['n_rows']:,} reviews: "
      f"{yesterday['accuracy_mean']:.4f} ± {yesterday['accuracy_std']:.4f}")
# TODO: the same TF-IDF pipeline on the same rows and folds, and print all three numbers.
# مهمة: خطّ TF-IDF نفسه على الصفوف والأثلام نفسها، واطبع الأرقام الثلاثة.
# TODO: and both predictions, and record whether the pair came out right.
# مهمة: وسجّل هل خرج الزوج صحيحًا.

## Section 3 — Stretch: semantic search in fifteen lines  (≈30 min)

Embed all 12,000 reviews, embed a free-text query, and return the top five by cosine. That is a
search engine, and it is one matrix product.

Then run the query that makes the point: **something whose words appear in none of the reviews you
want back.** `"the battery died after a week"` should retrieve complaints about power and charging
even from reviews that say "stopped holding a charge" and never use the word `battery`. TF-IDF
cannot do that: for every review that shares no term with the query its similarity is exactly
zero, and a ranking cannot order a field of zeros. Count how many of the 12,000 that is.

Compare the two side by side on the same query: TF-IDF's top five against the embeddings' top five.

This is W7D2 in miniature. In week 7 the matrix product is replaced by an index that does not have
to look at all 12,000 rows, the corpus is documents rather than reviews, and the top five get handed
to a language model — but the retrieval step is exactly what you just wrote.

<div dir="rtl" align="right">

## القسم الثالث — التمديد: بحث دلالي في خمسة عشر سطرًا (نحو ٣٠ دقيقة)

مثّل الاثنتي عشرة ألف مراجعة كلها، ومثّل استعلامًا نصيًا حرًّا، وأعِد أفضل خمسة بجيب التمام. فذلك
محرّك بحث، وهو جداء مصفوفات واحد.

ثم شغّل الاستعلام الذي يُثبت المقصود: **شيءٌ لا ترد كلماته في أي من المراجعات التي تريد استرجاعها.**
فـ`"the battery died after a week"` يجب أن يسترجع شكاوى عن الطاقة والشحن حتى من مراجعاتٍ تقول
«stopped holding a charge» ولا تستخدم كلمة `battery` قطّ. ولا يستطيع TF-IDF ذلك — فلا مصطلح مشترك،
ولا تشابه، ولا نتيجة.

قارن الاثنين جنبًا إلى جنب على الاستعلام نفسه: أفضل خمسة عند TF-IDF مقابل أفضل خمسة عند التمثيلات.

وهذا هو اليوم الثاني من الأسبوع السابع مصغّرًا. ففي الأسبوع السابع يُستبدل جداء المصفوفات بفهرسٍ لا
يحتاج أن ينظر في الصفوف الاثني عشر ألفًا كلها، وتكون المُدوّنة مستنداتٍ لا مراجعات، ويُسلَّم أفضل
خمسة إلى نموذج لغة — أما خطوة الاسترجاع فهي بعينها ما كتبته الآن.

</div>


In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Embedding 12,000 short reviews takes a minute or two on a CPU. Do it once and keep
#    the array — do not re-embed per query.
# 2) With normalize_embeddings=True the search is corpus @ query, then argsort.
# 3) For the TF-IDF comparison, fit a vectoriser on the reviews, transform the query,
#    and use the same top-k logic on the sparse similarities.
# 4) Print each result's similarity next to a 90-character excerpt so the two rankings
#    can be read against each other.
# Search: "semantic search cosine similarity top k sentence transformers"
# https://www.sbert.net/examples/applications/semantic-search/README.html
#
# ١) تمثيل اثنتي عشرة ألف مراجعة قصيرة يأخذ دقيقةً أو اثنتين على المعالج. افعله مرّةً
#    واحدة واحفظ المصفوفة — ولا تُعِد التمثيل لكل استعلام.
# ٢) وبـ`normalize_embeddings=True` يكون البحث `corpus @ query` ثم `argsort`.
# ٣) وللمقارنة بـTF-IDF، درّب مُتَّجِهًا على المراجعات، وحوّل الاستعلام، واستخدم منطق
#    أفضل `k` نفسه على التشابهات المُتفرّقة.
# ٤) اطبع تشابه كل نتيجة بجانب مقتطفٍ من تسعين محرفًا لتُقرأ الترتيبتان إحداهما مقابل
#    الأخرى.
# ابحث عن: "semantic search cosine similarity top k sentence transformers"
# https://www.sbert.net/examples/applications/semantic-search/README.html
# ────────────────────────────────────────────────────────────────────

SEARCH_QUERY = "the battery died after a week"
# TODO: cosine, and the top 5 that TF-IDF returns for the same query, side by side.
# مهمة: التمام، وأفضل خمسٍ يُعيدها TF-IDF للاستعلام نفسه، جنبًا إلى جنب.

## Save your artefact

`similarity_report.md` — the three warm-up cosines, the five neighbour lists, the analogies with the
one that failed, the measured bias gaps, and the three-way accuracy comparison.

It is markdown rather than JSON on purpose: half of what this lab produced is a judgement, and a
judgement needs a sentence. A5 asks for the embedding half of the write-up and this file is it.

<div dir="rtl" align="right">

## احفظ أثرك

`similarity_report.md` — جيوب التمام الثلاثة في الإحماء، وقوائم الجيران الخمس، والتناظرات ومعها
الذي فشل، وفجوات التحيّز المقيسة، ومقارنة الدقّة الثلاثية.

وهو ماركداون لا JSON عن قصد: فنصف ما أنتجه هذا المعمل حكمٌ، والحكم يحتاج جملة. ويطلب التكليف الخامس
نصفَ التمثيلات من التقرير، وهذا الملف هو ذلك النصف.

</div>


In [ ]:
lines = [
    "# W5D2 — Similarity report",
    "",
    "## Warm-up cosines",
    "",
    f"- `cosine([3,4], [4,3])` = **{COSINE_AB:.4f}**",
    f"- `cosine([3,4], [40,30])` = **{COSINE_SCALED:.4f}** (scaling changed it by "
    f"{abs(COSINE_AB - COSINE_SCALED):.1e})",
    f"- `cosine([3,4], [-4,3])` = **{COSINE_PERPENDICULAR:.4f}**",
    "",
    f"## The space",
    "",
    f"{E.shape[0]:,} tokens x {E.shape[1]} dimensions; row norms "
    f"{NORMS.min():.6f}–{NORMS.max():.6f}, so a dot product is a cosine.",
    f"Two PCA components keep {VARIANCE_KEPT:.1%} of the variance — {WHAT_PCA_PROVES}",
    "",
    "## Nearest neighbours",
    "",
]
for term, results in NEIGHBOURS.items():
    lines.append(f"- **{term}** — " + ", ".join(f"{t} ({s:.2f})" for t, s in results))
lines += [
    "",
    f"{SIMILAR_VERSUS_CONTEXT}",
    "",
    "## Analogies",
    "",
    "| analogy | expected | returned | verdict |",
    "|---|---|---|---|",
]
for result in ANALOGY_RESULTS:
    lines.append(f"| `{result['analogy']}` | {result['expected']} | "
                 f"{', '.join(result['returned'][:3])} | {result['verdict']} |")
lines += [
    "",
    f"Failed: {'; '.join(ANALOGY_FAILURES) or 'none'}",
    "",
    f"{WHAT_THE_FAILURE_SHOWS}",
    "",
    "## Measured bias",
    "",
    "| attributes | group A | group B | gap |",
    "|---|---|---|---|",
]
for name, values in BIAS_MEASUREMENT.items():
    lines.append(f"| {name} | {values['group_a']:.4f} | {values['group_b']:.4f} | "
                 f"{values['gap']:+.4f} |")
lines += [
    "",
    f"{DOWNSTREAM_CONSEQUENCE}",
    "",
    "## Embeddings against TF-IDF",
    "",
    "| model | reviews | accuracy |",
    "|---|---|---|",
    f"| embeddings + LogisticRegression | {SUBSET_SIZE} | {EMBEDDING_ACCURACY:.4f} |",
    f"| TF-IDF + LogisticRegression | {SUBSET_SIZE} | {TFIDF_SAME_DATA:.4f} |",
    f"| TF-IDF + LogisticRegression (D1) | {yesterday['n_rows']:,} | {TFIDF_FULL_DATA:.4f} |",
    "",
    f"On equal data the embeddings win by {EMBEDDING_ACCURACY - TFIDF_SAME_DATA:+.4f}. Against "
    f"D1's full-data baseline they lose by {EMBEDDING_ACCURACY - TFIDF_FULL_DATA:+.4f}: "
    f"{yesterday['n_rows'] // SUBSET_SIZE}x the training data outweighed the better representation.",
    "",
    f"Negation pair: cosine {NEGATION_COSINE:.3f}, classified correctly by the embedding model: "
    f"**{NEGATION_CORRECT}** (D1's TF-IDF: {yesterday['negation']['correct']}).",
    "",
]

REPORT_PATH = ARTEFACT_DIR / "similarity_report.md"
REPORT_PATH.write_text("\n".join(lines), encoding="utf-8")

print("\n".join(lines[:18]))
print(f"\n… wrote {REPORT_PATH.name} ({len(lines)} lines)")

## Sanity check

<div dir="rtl" align="right">

## فحص سلامة

</div>


In [ ]:
check_close(cosine(A, A), 1.0,
            "cosine(a, a) must be exactly 1 — a vector is perfectly similar to itself",
            "يجب أن يكون `cosine(a, a)` واحدًا تمامًا — فالمتّجه شبيه بنفسه شبهًا كاملًا",
            tol=1e-9)

check_close(cosine(A, -A), -1.0,
            "cosine(a, -a) must be exactly -1 — the opposite direction is the other end of the "
            "range, and a formula that cannot reach -1 is missing its sign",
            "يجب أن يكون `cosine(a, -a)` سالب واحد تمامًا — فالاتّجاه المعاكس هو الطرف الآخر "
            "للمدى، والصيغة التي لا تبلغ سالب واحد فقدت إشارتها",
            tol=1e-9)

check_close(COSINE_SCALED, COSINE_AB,
            f"scaling a vector by 10 must leave the cosine unchanged — got {COSINE_SCALED:.9f} "
            f"against {COSINE_AB:.9f}. If these differ you divided by the wrong norm",
            f"يجب أن يُبقي قياس متّجه بعشرة جيب التمام كما هو — والناتج {COSINE_SCALED:.9f} مقابل "
            f"{COSINE_AB:.9f}. فإن اختلفا فقد قسمت على المعيار الخاطئ",
            tol=1e-9)

similarity_values = [s for results in NEIGHBOURS.values() for _, s in results]
check(VECTORS_ARE_UNIT and all(-1.0 <= s <= 1.0 for s in similarity_values),
      f"every vector must be unit length and every similarity must land in [-1, 1] — norms span "
      f"{NORMS.min():.6f}–{NORMS.max():.6f} and similarities span {min(similarity_values):.4f}–"
      f"{max(similarity_values):.4f}",
      f"يجب أن يكون كل متّجه وحدوي الطول وأن يقع كل تشابه في `[-1, 1]` — المعايير تمتدّ "
      f"{NORMS.min():.6f}–{NORMS.max():.6f} والتشابهات {min(similarity_values):.4f}–"
      f"{max(similarity_values):.4f}")

duplicated = {term: results for term, results in NEIGHBOURS.items()
              if len({t for t, _ in results}) != len(results) or term in {t for t, _ in results}}
check(not duplicated,
      f"a neighbour list must have no duplicates and must not contain its own query — offending "
      f"terms: {sorted(duplicated)}",
      f"يجب ألّا تحوي قائمة الجيران تكرارًا ولا مُستعلَمها نفسه — والمصطلحات المخالفة: "
      f"{sorted(duplicated)}")

check(EMBEDDING_ACCURACY > TFIDF_SAME_DATA,
      f"on the same {SUBSET_SIZE} reviews and the same folds the embedding classifier must beat "
      f"TF-IDF — got {EMBEDDING_ACCURACY:.4f} against {TFIDF_SAME_DATA:.4f}. Compare it with "
      f"D1's {TFIDF_FULL_DATA:.4f} on 12,000 rows separately; that comparison is about data size",
      f"على المراجعات {SUBSET_SIZE} نفسها والأثلام نفسها يجب أن يتجاوز مُصنّف التمثيلات TF-IDF — "
      f"والناتج {EMBEDDING_ACCURACY:.4f} مقابل {TFIDF_SAME_DATA:.4f}. وقارنه بـ"
      f"{TFIDF_FULL_DATA:.4f} في اليوم الأول على اثني عشر ألف صف على حدة، فتلك مقارنة عن حجم "
      f"البيانات")

check(NEGATION_CORRECT and NEGATION_COSINE < 0.999,
      f"the embedding model should classify the negation pair correctly and the two sentences must "
      f"not be identical vectors — correct: {NEGATION_CORRECT}, cosine {NEGATION_COSINE:.4f}. If "
      f"the classification is wrong on your model, record that in the report rather than chasing "
      f"it: the vectors differing at all is the result that matters here",
      f"يُتوقّع أن يُصنّف نموذج التمثيل زوج النفي صحيحًا وألّا يكون للجملتين المتّجه نفسه — صحيح: "
      f"{NEGATION_CORRECT}، وجيب التمام {NEGATION_COSINE:.4f}. فإن كان التصنيف خاطئًا على نموذجك "
      f"فسجّل ذلك في التقرير ولا تُطارده: فاختلاف المتّجهين أصلًا هو النتيجة المهمّة هنا")

report_text = REPORT_PATH.read_text(encoding="utf-8")
required = ["Warm-up cosines", "Nearest neighbours", "Analogies", "Measured bias",
            "Embeddings against TF-IDF"]
missing = [heading for heading in required if heading not in report_text]
check(not missing and abs(FAMILY_GAP) > 0.01,
      f"similarity_report.md must carry all five sections and a measured bias gap — missing "
      f"{missing or 'nothing'}, and the family gap is {FAMILY_GAP:+.4f}",
      f"يجب أن يحمل `similarity_report.md` الأقسام الخمسة كلها وفجوة تحيّز مقيسة — الناقص "
      f"{missing or 'لا شيء'}، وفجوة الأسرة {FAMILY_GAP:+.4f}")

report()

## What's next

**W5D3 — Attention by hand.** Today's embeddings gave `the food was good but the service was bad`
and `the service was good but the food was bad` a cosine of about 0.995 — not identical, the way
TF-IDF made them, but not usefully different either. Word order is still almost invisible.

Tomorrow there is no dataset at all. Thirty lines of NumPy, three hard-coded vectors, and a
mechanism that lets one token's weight depend on another's — reproducing `[0.42, 0.16, 0.42]` and
`[0.84, 0.58]` from the morning slide. Then you prove that even *that* is order-blind on its own,
which is the reason Thursday exists.

<div dir="rtl" align="right">

## ما التالي

**الأسبوع ٥ اليوم ٣ — الانتباه باليد.** أعطت تمثيلات اليوم
`the food was good but the service was bad` و`the service was good but the food was bad` جيب تمامٍ
نحو ٠٫٩٩٥ — لا متطابقتين كما جعلهما TF-IDF، ولا مختلفتين اختلافًا مُفيدًا. فترتيب الكلمات ما زال
يكاد لا يُرى.

وغدًا لا توجد بيانات أصلًا. ثلاثون سطرًا من NumPy، وثلاثة متّجهات مكتوبة بثبوت، وآليّة تجعل وزن رمزٍ
يتوقّف على آخر — مُعيدةً إنتاج `[0.42, 0.16, 0.42]` و`[0.84, 0.58]` من شريحة الصباح. ثم تُبرهن أن
**تلك** أيضًا عمياء عن الترتيب وحدها، وهذا سبب وجود الخميس.

</div>
